# Interpretable morphological features

The supplied cleaned well aggregates are plotted directly. Outliers are defined against pooled WT median/MAD; density contours and the source exclusion lists are preserved. Metadata and cell counts do not enter the feature axes.

In [ ]:
from pathlib import Path
import numpy as np
import numpy.typing
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PACKAGE = Path.cwd()
if not (PACKAGE / "Datasets").is_dir():
    PACKAGE = PACKAGE.parent
DATASETS = PACKAGE / "Datasets"
if not DATASETS.is_dir():
    raise FileNotFoundError("Launch this notebook from the package root or Github folder.")
plt.rcParams.update({"font.family": "Arial", "text.usetex": False})

rescaled_df = pd.read_excel(DATASETS / "Dataset S3.xlsx", sheet_name="Raw_well_profiles")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from scipy.stats import spearmanr

def _darken_color(color, amount=0.35):
    (r, g, b) = to_rgb(color)
    return (r * (1 - amount), g * (1 - amount), b * (1 - amount))

def _robust_zscore(values, median, mad, scale=1.4826):
    if mad == 0 or np.isnan(mad):
        return np.full(len(values), np.nan)
    return np.abs((values - median) / (scale * mad))

def plot_outliers_with_labels_and_contours(df, features, metadata_gene_col='Metadata_gene', threshold=3, xlabel=None, ylabel=None, exclude_genes=None, genes_to_label=None, base_color='#0072B2', outlier_darken=0.35, point_size=6, alpha_points=0.8, contour_levels=8, show_labels=True, use_threshold=True, ax=None, show_legend=True, show_spearman=True, spearman_pos=(0.03, 0.97), spearman_fontsize=8, ylim=None, print_deviation_summary=True):
    (feature_x, feature_y) = features
    df_filtered = df[np.isfinite(df[feature_x]) & np.isfinite(df[feature_y])].copy()
    if exclude_genes:
        df_filtered = df_filtered[~df_filtered[metadata_gene_col].isin(exclude_genes)]
    if df_filtered.empty:
        return (None, None, None)
    wt_data = df_filtered[df_filtered[metadata_gene_col] == 'WT'].copy()
    if wt_data.empty:
        return (None, None, None)
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'axes.linewidth': 0.6})
    base = base_color
    outlier_color = _darken_color(base, outlier_darken)
    deviation_summary = None
    if use_threshold and threshold is not None:
        wt_median_x = wt_data[feature_x].median()
        wt_median_y = wt_data[feature_y].median()
        wt_mad_x = np.median(np.abs(wt_data[feature_x] - wt_median_x))
        wt_mad_y = np.median(np.abs(wt_data[feature_y] - wt_median_y))
        df_filtered['rz_x'] = _robust_zscore(df_filtered[feature_x].to_numpy(), wt_median_x, wt_mad_x)
        df_filtered['rz_y'] = _robust_zscore(df_filtered[feature_y].to_numpy(), wt_median_y, wt_mad_y)
        df_filtered['deviates_x'] = df_filtered['rz_x'] > threshold
        df_filtered['deviates_y'] = df_filtered['rz_y'] > threshold
        df_filtered['deviates_either'] = df_filtered['deviates_x'] | df_filtered['deviates_y']
        outliers = df_filtered[df_filtered['deviates_either']].copy()
        mutant_df = df_filtered[df_filtered[metadata_gene_col] != 'WT'].copy()
        gene_dev = mutant_df.groupby(metadata_gene_col, as_index=False).agg(deviates_x=('deviates_x', 'max'), deviates_y=('deviates_y', 'max'), deviates_either=('deviates_either', 'max'))
        n_mutants = len(gene_dev)
        deviation_summary = pd.DataFrame({'feature': [feature_x, feature_y, 'either feature'], 'n_deviating_mutants': [int(gene_dev['deviates_x'].sum()), int(gene_dev['deviates_y'].sum()), int(gene_dev['deviates_either'].sum())], 'n_total_mutants': n_mutants})
        deviation_summary['pct_deviating_mutants'] = 100 * deviation_summary['n_deviating_mutants'] / n_mutants
    else:
        outliers = pd.DataFrame(columns=df_filtered.columns)
    label_rows = outliers.copy()
    if genes_to_label:
        forced = df_filtered[df_filtered[metadata_gene_col].isin(genes_to_label)]
        label_rows = pd.concat([label_rows, forced]).drop_duplicates()
    created_fig = False
    if ax is None:
        (fig, ax) = plt.subplots(figsize=(4, 3))
        created_fig = True
    else:
        fig = ax.figure
    ax.scatter(df_filtered[feature_x], df_filtered[feature_y], s=point_size, c=base, alpha=alpha_points, linewidths=0)
    if not outliers.empty:
        ax.scatter(outliers[feature_x], outliers[feature_y], s=point_size, c=[outlier_color], alpha=0.9, linewidths=0)
    wt_mass = np.array([0.1, 0.3, 0.5, 0.7, 0.9])
    kde_levels = np.sort(1 - wt_mass)
    sns.kdeplot(x=wt_data[feature_x], y=wt_data[feature_y], ax=ax, color='k', fill=False, alpha=0.5, levels=kde_levels, linewidths=0.6)
    if show_labels:
        for (_, row) in label_rows.iterrows():
            ax.text(row[feature_x], row[feature_y], str(row[metadata_gene_col]), fontsize=8, ha='left', va='center')
    ax.set_xlabel(xlabel if xlabel else feature_x.replace('_', ' '))
    ax.set_ylabel(ylabel if ylabel else feature_y.replace('_', ' '))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(direction='out', length=3, width=0.9)
    ax.set_title(f"{feature_x.replace('_', ' ')} vs {feature_y.replace('_', ' ')}", fontweight='bold')
    if ylim is not None:
        ax.set_ylim(ylim)
    if show_spearman and len(df_filtered) >= 2:
        (rho, pval) = spearmanr(df_filtered[feature_x], df_filtered[feature_y], nan_policy='omit')
        if np.isfinite(rho):
            ax.text(spearman_pos[0], spearman_pos[1], f'Spearman r = {rho:.2f}', transform=ax.transAxes, ha='left', va='top', fontsize=spearman_fontsize, bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1.5))
    if show_legend:
        handles = [Line2D([0], [0], marker='o', color='none', markerfacecolor=base, markersize=5, alpha=alpha_points), Line2D([0], [0], marker='o', color='none', markerfacecolor=outlier_color, markersize=5, alpha=0.9), Line2D([0], [0], color='black', lw=0.9)]
        leg_labels = ['Mutants', 'Outliers', 'WT density']
        if not use_threshold:
            leg_labels[1] = 'Highlighted'
        ax.legend(handles, leg_labels, frameon=False, loc='upper right', fontsize=8)
    if created_fig:
        fig.tight_layout()
        plt.show()
    return (df_filtered, outliers, deviation_summary)

In [ ]:
plot_outliers_with_labels_and_contours(rescaled_df, show_labels=False, features=['cell_length', 'cell_width'], metadata_gene_col='Metadata_gene', xlabel='Average Cell Length (μm)', ylabel='Average Cell Width (μm)', base_color='#9D9D9D', threshold=4, genes_to_label=['ynfH', 'ssnA', 'yfjK'], exclude_genes=['yhjJ', 'xdhA', 'yqeC', 'ygeW', 'yijE', 'ygeM', 'yhdY', 'ygfK', 'pgi', 'ypdA', 'yehE', 'ydhT', 'yajR', 'gntY', 'yigL', 'yobG', 'dnaT'])
plot_outliers_with_labels_and_contours(rescaled_df, show_labels=False, features=['C5_std_object_avg_center_distance', 'cell_volume'], metadata_gene_col='Metadata_gene', ylabel='Average Cell Volume (μm³)', base_color='#77BBFF', threshold=4, xlabel='Standard Deviation of Relative Nucleoid Off-center Distance', exclude_genes=['yhjJ', 'xdhA', 'yqeC', 'ygeW', 'yijE', 'ygeM', 'yhdY', 'ygfK', 'pgi', 'ypdA', 'yehE', 'ydhT', 'yajR', 'gntY', 'yigL', 'yobG', 'dnaT'])
plot_outliers_with_labels_and_contours(rescaled_df, show_labels=False, features=['cell_length_robust_CV', 'cell_width_robust_CV'], metadata_gene_col='Metadata_gene', base_color='#8682A5EB', threshold=4, exclude_genes=['yhjJ', 'xdhA', 'yqeC', 'ygeW', 'yijE', 'ygeM', ' yhdY', ' ygfK ', 'pgi ', 'ypdA ', ' yehE ', 'ydhT ', 'yajR ', 'gntY ', 'yigL ', 'yobG ', 'dnaT'])
plot_outliers_with_labels_and_contours(rescaled_df, show_labels=False, features=['C5_NC_ratio', 'C4_NC_ratio'], metadata_gene_col='Metadata_gene', ylabel='RNA-to-Cell Area Ratio', xlabel='Nucleoid-to-Cell Area Ratio', base_color='#3CE79AC5', threshold=4, exclude_genes=['yhjJ', 'xdhA', 'yqeC', 'ygeW', 'yijE', 'ygeM', 'yhdY', 'ygfK', 'pgi', 'ypdA', 'yehE', 'ydhT', 'yajR', 'gntY', 'yigL', 'yobG', 'dnaT'])
plot_outliers_with_labels_and_contours(rescaled_df, show_labels=False, features=['C2_NC_ratio_CV', 'cell_volume'], metadata_gene_col='Metadata_gene', xlabel='OM heterogeneity', ylabel='Cell Volume', base_color='#FC7E7EFF', threshold=4, exclude_genes=['yhjJ ', ' xdhA ', ' yqeC ', ' ygeW ', ' yijE ', ' ygeM ', ' yhdY ', ' ygfK ', 'pgi ', 'ypdA ', ' yehE ', 'ydhT ', 'yajR ', 'gntY ', 'yigL ', 'yobG ', 'dnaT'])
plot_outliers_with_labels_and_contours(rescaled_df, show_labels=False, features=['C3_NC_ratio_CV', 'cell_volume'], metadata_gene_col='Metadata_gene', xlabel='IM heterogeneity', ylabel='Cell Volume', base_color='#FF92F6E1', threshold=4, exclude_genes=['yhjJ ', ' xdhA ', ' yqeC ', ' ygeW ', ' yijE ', ' ygeM ', ' yhdY ', ' ygfk ', 'pgi ', 'ypdA ', ' yehE ', 'ydhT ', 'yajR ', 'gntY ', 'yigL ', 'yobG ', 'dnaT'])
plot_outliers_with_labels_and_contours(rescaled_df, show_labels=False, features=['cell_area', 'C5_cell_total_obj_area'], metadata_gene_col='Metadata_gene', xlabel='Cell area', ylabel='Nucleoid area', base_color='#94E4EDC7', threshold=None, exclude_genes=['yhjJ ', ' xdhA ', ' yqeC ', ' ygeW ', ' yijE ', ' ygeM ', ' yhdY ', ' ygfk ', 'pgi ', 'ypdA ', ' yehE ', 'ydhT ', 'yajR ', 'gntY ', 'yigL ', 'yobG ', 'dnaT'])
plot_outliers_with_labels_and_contours(rescaled_df, show_labels=False, features=['cell_area', 'C4_cell_total_obj_area'], metadata_gene_col='Metadata_gene', xlabel='Cell area', ylabel='RNA area', base_color='#6EE2B3DF', threshold=None, exclude_genes=['yhjJ ', ' xdhA ', ' yqeC ', ' ygeW ', ' yijE ', ' ygeM ', ' yhdY ', ' ygfk ', 'pgi ', 'ypdA ', ' yehE ', 'ydhT ', 'yajR ', 'gntY ', 'yigL ', 'yobG ', 'dnaT'])
plot_outliers_with_labels_and_contours(rescaled_df, show_labels=False, ylim=(0, 1), features=['cell_area', 'C4_NC_ratio'], metadata_gene_col='Metadata_gene', xlabel=None, ylabel=None, base_color='#6EE2B3DF', threshold=None, exclude_genes=['yhjJ ', ' xdhA ', ' yqeC ', ' ygeW ', ' yijE ', ' ygeM ', ' yhdY ', ' ygfk ', 'pgi ', 'ypdA ', ' yehE ', 'ydhT ', 'yajR ', 'gntY ', 'yigL ', 'yobG ', 'dnaT'])
plot_outliers_with_labels_and_contours(rescaled_df, show_labels=False, ylim=(0, 1), features=['C4_cell_total_obj_area', 'C4_NC_ratio'], metadata_gene_col='Metadata_gene', xlabel=None, ylabel=None, base_color='#6EE2B3DF', threshold=None, exclude_genes=['yhjJ ', ' xdhA ', ' yqeC ', ' ygeW ', ' yijE ', ' ygeM ', ' yhdY ', ' ygfk ', 'pgi ', 'ypdA ', ' yehE ', 'ydhT ', 'yajR ', 'gntY ', 'yigL ', 'yobG ', 'dnaT'])
plot_outliers_with_labels_and_contours(rescaled_df, show_labels=False, ylim=(0, 1), features=['cell_area', 'C5_NC_ratio'], metadata_gene_col='Metadata_gene', xlabel=None, ylabel=None, base_color='#94E4EDC7', threshold=None, exclude_genes=['yhjJ ', ' xdhA ', ' yqeC ', ' ygeW ', ' yijE ', ' ygeM ', ' yhdY ', ' ygfk ', 'pgi ', 'ypdA ', ' yehE ', 'ydhT ', 'yajR ', 'gntY ', 'yigL ', 'yobG ', 'dnaT'])
plot_outliers_with_labels_and_contours(rescaled_df, show_labels=False, ylim=(0, 1), features=['C5_cell_total_obj_area', 'C5_NC_ratio'], metadata_gene_col='Metadata_gene', xlabel=None, ylabel=None, base_color='#94E4EDC7', threshold=None, exclude_genes=['yhjJ ', ' xdhA ', ' yqeC ', ' ygeW ', '\tyijE ', '\tygeM ', '\tyhdY ', '\tygfK ', 'pgi ', 'ypdA ', '\tyehE ', '\tydhT ', '\tyajR ', '\tgntY ', '\tyigL ', '\tyobG ', '\tdnaT'])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
plt.rcParams.update({'font.family': 'Arial', 'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'axes.linewidth': 0.6})

def plot_two_features_colored_by_third(df, x_feature, y_feature, color_feature, metadata_gene_col='Metadata_gene', exclude_genes=None, genes_to_label=None, clip_percentiles=(1, 99), cmap='cividis', point_size=8, alpha=0.85, ylim=None, show_wt_contours=True, contour_levels=8, show_spearman=True, figsize=(4, 3)):
    """
    Scatter plot:
        x-axis = x_feature
        y-axis = y_feature
        color = color_feature, with colorbar
    """
    required = [x_feature, y_feature, color_feature, metadata_gene_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Missing columns: {missing}')
    plot_df = df[np.isfinite(df[x_feature]) & np.isfinite(df[y_feature]) & np.isfinite(df[color_feature])].copy()
    if exclude_genes is not None:
        plot_df = plot_df[~plot_df[metadata_gene_col].isin(exclude_genes)].copy()
    if plot_df.empty:
        raise ValueError('No valid rows left after filtering.')
    color_values = pd.to_numeric(plot_df[color_feature], errors='coerce')
    if clip_percentiles is not None:
        (vmin, vmax) = np.nanpercentile(color_values, clip_percentiles)
        color_values_plot = color_values.clip(vmin, vmax)
    else:
        (vmin, vmax) = (np.nanmin(color_values), np.nanmax(color_values))
        color_values_plot = color_values
    (fig, ax) = plt.subplots(figsize=figsize)
    sc = ax.scatter(plot_df[x_feature], plot_df[y_feature], c=color_values_plot, cmap=cmap, vmin=vmin, vmax=vmax, s=point_size, alpha=alpha, linewidths=0, rasterized=True, zorder=10)
    if show_wt_contours and metadata_gene_col in plot_df.columns:
        wt_df = plot_df[plot_df[metadata_gene_col].eq('WT')]
        wt_mass = np.array([0.1, 0.3, 0.5, 0.7, 0.9])
        kde_levels = np.sort(1 - wt_mass)
        if len(wt_df) > 10:
            sns.kdeplot(x=wt_df[x_feature], y=wt_df[y_feature], ax=ax, color='black', fill=False, alpha=0.55, levels=kde_levels, linewidths=0.6, zorder=20)
    if genes_to_label is not None:
        label_df = plot_df[plot_df[metadata_gene_col].isin(genes_to_label)]
        for (_, row) in label_df.iterrows():
            ax.text(row[x_feature], row[y_feature], str(row[metadata_gene_col]), fontsize=7, ha='left', va='center', zorder=30)
    if show_spearman and len(plot_df) >= 3:
        (rho, pval) = spearmanr(plot_df[x_feature], plot_df[y_feature], nan_policy='omit')
        if np.isfinite(rho):
            ax.text(0.03, 0.97, f'Spearman r = {rho:.2f}', transform=ax.transAxes, ha='left', va='top', fontsize=8, bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=1.5))
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.set_xlabel(x_feature.replace('_', ' '))
    ax.set_ylabel(y_feature.replace('_', ' '))
    ax.set_title(f"{x_feature.replace('_', ' ')} vs {y_feature.replace('_', ' ')}", fontweight='bold', pad=6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(direction='out', length=3, width=0.8)
    cbar = fig.colorbar(sc, ax=ax, pad=0.02)
    cbar.set_label(color_feature.replace('_', ' '))
    cbar.outline.set_linewidth(0.6)
    cbar.ax.tick_params(length=2, width=0.6, labelsize=7)
    fig.tight_layout()
    plt.show()
    return plot_df
plot_df = plot_two_features_colored_by_third(df=rescaled_df, x_feature='C2_NC_ratio_CV', y_feature='C3_NC_ratio_CV', color_feature='cell_width_robust_CV', cmap='YlOrRd', genes_to_label=None)

In [ ]:
plot_df = plot_two_features_colored_by_third(df=rescaled_df, x_feature='cell_area', y_feature='C5_cell_total_obj_area', color_feature='C5_NC_ratio', cmap='YlGnBu', genes_to_label=None)

In [ ]:
plot_df = plot_two_features_colored_by_third(df=rescaled_df, x_feature='cell_area', y_feature='C4_cell_total_obj_area', color_feature='C4_NC_ratio', cmap='YlGnBu', genes_to_label=None)